# SketClothes — Train on Kaggle

**FashionSD-X pipeline:** SD 1.5 + LoRA (stage 1) + ControlNet sketch (stage 2)

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Run all cells

In [ ]:
# Clone repo 
!git clone https://github.com/lqb464/SketClothes.git
%cd /kaggle/working/SketClothes
!pip install -q -U -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
os.environ["DATASET_ID"] = "/kaggle/input/datasets/lqb464/sketclothes-data"

# Stage 1 + 2 training
!python training/train_all.py

In [ ]:
# Test inference sau khi train hoàn tất
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DPMSolverMultistepScheduler
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

base_model = "runwayml/stable-diffusion-v1-5"
lora_dir = "/kaggle/working/outputs/lora"
controlnet_dir = "/kaggle/working/outputs/controlnet"

print("[+] Loading trained ControlNet...")
controlnet = ControlNetModel.from_pretrained(controlnet_dir, torch_dtype=torch.float16)

print("[+] Building Stable Diffusion ControlNet Pipeline...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

print("[+] Loading Stage 1 Fashion LoRA...")
pipe.load_lora_weights(lora_dir, weight_name="pytorch_lora_weights.safetensors")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=False)

# Chọn sketch để test
dataset_dir = Path("/kaggle/input/datasets/lqb464/sketclothes-data")
sketch_path = dataset_dir / "sketches/000010.png"  # thay đổi đường dẫn tới file sketch bạn muốn test

if sketch_path.exists():
    sketch_img = Image.open(sketch_path).convert("RGB").resize((512, 512))
else:
    print(f"[!] Không tìm thấy {sketch_path}, dùng ảnh trắng. Hãy sửa sketch_path.")
    sketch_img = Image.new("RGB", (512, 512), (255, 255, 255))

prompt = "a high quality fashion photo of a beautiful evening gown, detailed fabric texture, professional studio lighting, clean white background"
negative_prompt = "low quality, blurry, distorted, bad proportions, watermark, text"

print(f"\n[+] Generating image for prompt: '{prompt}'...")
result_img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=sketch_img,
    num_inference_steps=25,
    guidance_scale=7.5,
    generator=torch.Generator(device="cuda").manual_seed(42)
).images[0]

# Hiển thị so sánh
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sketch_img)
axes[0].set_title("Sketch Input")
axes[0].axis("off")

axes[1].imshow(result_img)
axes[1].set_title("Generated Fashion Garment")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/test_result.png", dpi=150)
plt.show()
print("[+] Đã lưu kết quả tại /kaggle/working/test_result.png")